# 어휘 홀드아웃 평가 — 11차 레시피가 한 번도 안 본 단어에서 어떻게 하는가

**질문**: 11차 통제군 평가에서 단어 이득의 26%만 어휘 범위 확장에서 왔다(대장 「어휘
분해 결과」). 그 프록시는 118개(30종)뿐이고 통제군 쪽 사정에 좌우됐다. 이번엔
**11차 레시피 자체가** 한 번도 학습에서 못 본 단어(15종, `build_vocab_holdout.py`로
train·dev·test 전부에서 뺐다)에서 12차를 재고, 같은 단어에서 11차와 직접 비교한다.

**⚠️ 주 비교는 12차 대 11차로 한정한다.** 15종 중 14종이 이미 big6 학습 어휘에도
있어서(7차재현·10차는 "처음 보는 단어"가 아니다), 통제군 비교는 대부분 오염된다.
넷 다 공정한 단어는 `사탕` 하나뿐이다(대장 「12차 계획」 참고). 7차재현·10차는
참고용으로만 신는다.

**판정 규칙(결과 보기 전에 대장에 적었다)**: 원래 dev·test였던 행(457개, 어느
모델도 그 클립은 학습에 안 씀)에서 12차의 단어 CER이 11차보다 0.10 이내로만
나쁘면 "어휘 몰라도 봐줄 만하다". 원래 train이었던 행(1183개)은 11차에게 이미
본 데이터라 비교에서 뺀다 — 12차 자신의 절대 성능에만 쓴다.

- 셀 1~3 · 6 · 7은 이전 노트북(`eval_adapters_big7_colab.ipynb`)의 셀을 **그대로**
  가져왔다. 셀 3의 ZIP만 `colab_trainset_big7_vochold.zip`으로 바뀌었다.
- 셀 4는 `ADAPTERS`에 `12차`를 추가했을 뿐 나머지는 같다.
- 준비물: 드라이브에 `colab_trainset_big7_vochold.zip`과 `whisper-608-lora-12차`,
  `-11차`(필수), `-7차재현`·`-10차`(참고용, 없으면 건너뛴다).


## 셀 1 — GPU 확인 (Tesla T4가 떠야 함)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (환경 핀). numpy가 바뀔 때만 재시작을 요구한다

In [ ]:
# ══ 환경 고정 ═══════════════════════════════════════════════
# 회차 간 비교를 하려면 프레임워크가 회차마다 바뀌면 안 된다. 7차(8/18)는
# transformers 4.x, 7차재현(8/20)은 5.15.1이었고 그 사이에 같은 배치 설정이
# T4에서 OOM으로 죽었다. 올릴 때는 회차를 새로 시작하고 대장에 환경을 적는다.
TFV = "5.15.1"

# numpy는 Colab 이미지가 든 버전을 그대로 고정한다. 숫자를 하드코딩하면
# 이미지가 바뀔 때마다 다시 깨진다:
#   너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy, subprocess, sys
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정 / transformers →", TFV)

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U "transformers=={TFV}" datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

# **새 프로세스로** 확인한다. 이 커널이 이미 들고 있는 모듈은 pip이 디스크를
# 바꿔도 안 바뀌므로, 커널 안에서 import해 확인하면 거짓 통과한다.
_out = subprocess.run(
    [sys.executable, "-c",
     "import numpy, transformers; print(numpy.__version__, transformers.__version__)"],
    capture_output=True, text=True)
print("디스크:", (_out.stdout or _out.stderr).strip())
_disk = _out.stdout.split()

if len(_disk) != 2 or _disk[1] != TFV:
    raise SystemExit(f"transformers가 {TFV}가 아니다. 위 pip 출력을 확인해라.")
# numpy만이 재시작을 강제한다 — 이 커널이 이미 로드했기 때문이다.
# transformers·datasets 등은 아직 import 전이라 새 파일이 그대로 읽힌다.
if _disk[0] != NPV:
    raise SystemExit(f"numpy가 {NPV} → {_disk[0]}로 바뀌었다."
                     " [런타임 → 세션 다시 시작] 후 셀 2.5부터 다시 실행해라.")
print("✅ 설치 완료 — 재시작 없이 셀 2.5로 계속 진행")


## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
if not transformers.__version__.startswith("5.15"):
    raise SystemExit("transformers 핀이 안 먹었다 — 셀 2를 다시 실행해라.")

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 + 업로드한 zip 풀기 + 경로 확인
`ZIP`을 본인이 올린 위치로 수정.

## 셀 3 — 드라이브 마운트 + vochold zip 풀기

In [ ]:
import os, torch, json
from google.colab import drive
drive.mount('/content/drive')

ZIP  = "/content/drive/MyDrive/colab_trainset_big7_vochold.zip"   # ← 업로드한 zip 경로
# big7_vochold = 12차·이 노트북 전용(build_vocab_holdout.py 산출). 단어 15종을
# train·dev·test 전부에서 빼고, 그 단어의 모든 클립을 holdout_eval.jsonl에 모았다.
# big7 = VS01 병합본(11차부터). train 2281→10485, dev 326→1427, test 417→2673 —
# **train만 4.6배가 아니라 dev·test도 커졌다.** 그래서 옛 dev/test 수치(0.3101 등)를
# 그대로 인용하면 안 된다 — 시험지 자체가 다른 시험지다(아래 셀 8 참고).
BASE = "/content/608-trainset"                       # 풀 위치(세션 로컬, 빠름)
if os.path.exists(ZIP):
    !unzip -oq "$ZIP" -d "$BASE"
else:
    # zip 대신 폴더째 올렸으면 그 경로를 BASE로 지정하고 위 unzip은 건너뛴다
    BASE = "/content/drive/MyDrive/608-trainset"

print("CUDA:", torch.cuda.is_available())

# dev는 선택이 아니다. dev 없이 돌리면 test로 체크포인트를 고르게 되고, 그러면
# 보고되는 CER이 홀드아웃 성능이 아니라 "시험지를 보며 고른 점수"가 된다.
# 4~6차 배치가 그 상태였다. 없으면 여기서 멈추고 재패키징을 요구한다.
if not os.path.exists(os.path.join(BASE, "holdout_eval.jsonl")):
    raise SystemExit(
        "holdout_eval.jsonl이 없다 — big7_vochold.zip이 아니라 big7.zip을 올린 것 아닌지 확인해라."
    )

for f in ["train.jsonl","dev.jsonl","test.jsonl","holdout_eval.jsonl"]:
    p=os.path.join(BASE,f); n=sum(1 for _ in open(p,encoding="utf-8"))
    print(f"  {f}: {n}개")
print("wav:", sum(len(fs) for _,_,fs in os.walk(os.path.join(BASE,'wav'))), "개")

## 셀 4 — 모델 + 어댑터 로드 (12차 포함)

In [ ]:
# 할당자가 크기별로 굳은 세그먼트를 재사용하지 못해 여유가 있는데도 OOM이 나는
# 경우가 있다(생성 평가는 배치마다 시퀀스 길이가 달라 특히 그렇다). 이 설정은
# **CUDA 컨텍스트가 만들어지기 전에** 걸려야 효과가 있어 맨 위에 둔다.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

# 베이스 모델. zero-shot 비교(셀 10) 결과로 large-v3-turbo를 채택했다.
#
#   모델              단어 CER   문장 CER
#   small               2.000      0.404
#   medium              1.376      0.363
#   large-v3-turbo      0.890      0.321   ← 두 지표 모두 승
#
# 단어가 모델 크기에 훨씬 민감하다(2.2배 개선 vs 문장 1.26배). 앱 최약점이
# 단어 수준 구음장애라 이 차이가 그대로 제품 성능이다. turbo는 디코더가 얕아
# large-v3(1550M)보다 훨씬 빠르면서 T4에 들어간다.
#
# 되돌리려면 이 한 줄만 바꾼다. 프로세서도 같은 ID로 로드되므로 mel 빈 수
# (large-v3 계열 128 vs small/medium 80)가 자동으로 맞는다 — 프로세서를
# 재사용하면 입력이 조용히 깨진다.
MODEL_ID = "openai/whisper-large-v3-turbo"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
# **fp32로 명시 로드해야 한다.** large-v3-turbo는 config에 torch_dtype: float16이
# 박혀 있어 최신 transformers가 fp16 가중치로 올린다. 그러면 특징 추출기가 내놓는
# fp32 입력과 어긋나 인코더 conv에서 터진다:
#   RuntimeError: Input type (float) and bias type (c10::Half) should be the same
# small은 config가 fp32라 이 문제가 없어 지금까지 안 드러났다.
#
# 혼합정밀은 Trainer의 fp16=True(autocast)가 맡는다. 가중치까지 fp16으로 두고
# 학습하면 마스터 가중치가 없어 불안정하다. T4는 Turing이라 bf16이 없어 이 조합이
# 유일한 선택지다.
# **attn_implementation을 명시한다.** 기본값에 맡기면 transformers 버전에 따라
# eager로 떨어질 수 있고, 그러면 어텐션 확률 행렬이 통째로 저장된다:
#   배치4 x 헤드20 x 1500 x 1500 x 2B = 360MB/층  x  인코더 32층 = 11.5GB
# sdpa는 그걸 아예 만들지 않는다. 이 한 줄이 T4에 들어가느냐 마느냐를 가른다.
# (셀 2가 transformers를 핀 없이 -U로 깔기 때문에 기본값에 기댈 수 없다.)
try:
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, attn_implementation='sdpa')
except (ValueError, ImportError) as e:
    print('sdpa 거부됨:', e)
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
print('어텐션 구현:', getattr(model.config, '_attn_implementation', '?'),  # 'sdpa'여야 한다
      '| torch', torch.__version__)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# ══ 어댑터 로드 — 학습된 LoRA를 붙인다(새로 만들지 않는다) ═══════════════
# 원본 셀 4는 여기서 get_peft_model(LoraConfig(...))로 **새** LoRA를 만든다.
# 통제군 평가는 학습을 안 하므로 저장된 어댑터를 읽는다. 어댑터 여러 개를 한
# 모델에 얹어 set_adapter로 바꿔 가며 잰다(베이스 3.2GB를 한 번만 올린다).
from peft import PeftModel

# 평가 순서. 11차를 먼저 재는 이유: 학습 중 기록된 dev 값과 이 노트북의 값이
# 맞는지(=평가 경로가 같은지) 가장 먼저 확인해야 나머지 숫자를 믿을 수 있다.
ADAPTERS = ["12차", "11차", "7차재현", "10차"]   # 12차·11차는 필수, 나머지는 참고용(없으면 건너뜀)
ADAPTER_DIR = "/content/drive/MyDrive/whisper-608-lora-{}"

found, absent = [], []
for _n in ADAPTERS:
    (found if os.path.isdir(ADAPTER_DIR.format(_n)) else absent).append(_n)
print("어댑터 있음:", found)
if absent:
    print("어댑터 없음:", absent, "→ 이 회차는 건너뛴다(재학습 없이는 통제군이 못 된다)")
if not found:
    raise SystemExit("드라이브에 어댑터가 하나도 없다 — whisper-608-lora-<회차> 폴더를 확인해라.")

model = PeftModel.from_pretrained(model, ADAPTER_DIR.format(found[0]), adapter_name=found[0])
for _n in found[1:]:
    model.load_adapter(ADAPTER_DIR.format(_n), adapter_name=_n)
model.eval()
print("로드 완료. 활성:", model.active_adapter)

# 생성(평가) 설정 — 이거 없으면 whisper가 반복 환각으로 CER이 2.0+로 부풀려진다.
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.no_repeat_ngram_size = 3   # 같은 구절 반복 억제

# ── 평가 시간의 대부분은 여기서 결정된다 ─────────────────────────
# 자기회귀 생성이라 **배치 안에서 가장 긴 시퀀스**가 그 배치의 시간을 정한다.
# 그리고 학습 전/초기 모델은 환각으로 거의 항상 상한까지 뱉는다 — 즉 상한값이
# 곧 평가 비용이다.
#
# 실측 라벨 길이 (colab_trainset_big6):
#     train 단어  중앙 1자 · 최대  2자
#     train 문장  중앙 13자 · p95 34자 · 최대 93자
#     dev  문장  최대 56자   |   test 문장 최대 44자
#
# 100토큰은 dev/test 최장(56자)의 두 배 가까이라 낭비다. 72로 낮춰 환각 꼬리를
# 자른다. 잘림 위험은 없다 — 한국어는 음절당 1토큰 안팎이라 56자면 60토큰 미만이다.
model.generation_config.max_new_tokens = 72

# **KV 캐시를 반드시 켠다.** 위에서 config.use_cache=False로 둔 건 학습용 관례인데,
# 그게 생성까지 따라가면 매 토큰마다 전체 시퀀스를 재계산해 평가가 몇 배 느려진다.
# (gradient_checkpointing을 안 쓰므로 학습에도 use_cache=False가 필요 없지만,
#  건드리면 학습 조건이 바뀌므로 생성 쪽만 명시적으로 켠다.)
model.generation_config.use_cache = True


## 셀 5a — 데이터 (holdout_eval만)

In [ ]:
import collections
from datasets import load_dataset, Audio

# holdout_eval.jsonl만 본다 — build_vocab_holdout.py가 만든, 15종의 모든 클립을
# 원래 스플릿(orig_split)과 함께 모아 둔 파일이다. train/dev/test는 이 실험에서
# 체크포인트 선택용으로만 쓰이고(셀 8), 여기서는 다시 안 읽는다.
if not os.path.exists(f"{BASE}/holdout_eval.jsonl"):
    raise SystemExit(
        "holdout_eval.jsonl이 없다 — big7 zip을 잘못 올렸을 수 있다. "
        "colab_trainset_big7_vochold.zip인지 확인해라."
    )

raw = [json.loads(l) for l in open(f"{BASE}/holdout_eval.jsonl", encoding="utf-8")]
words = [r["text"].strip() for r in raw]
orig_split = [r["orig_split"] for r in raw]

ds = load_dataset("json", data_files={"holdout": f"{BASE}/holdout_eval.jsonl"})
ds = ds.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))


def prep(b):   # eval_adapters_big7_colab.ipynb의 prep과 글자 그대로 같다
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b


ds = ds.map(prep, remove_columns=ds["holdout"].column_names, writer_batch_size=20)

assert len(ds["holdout"]) == len(words) == len(orig_split), "행 수가 안 맞는다"
print(ds)
print("단어 종류", len(set(words)), "개 ·", dict(collections.Counter(orig_split)))


## 셀 6 — 데이터 콜레이터 (whisper 필수)

In [ ]:
from dataclasses import dataclass
from typing import Any
@dataclass
class Collator:
    processor: Any
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:,0] == self.processor.tokenizer.bos_token_id).all():
            lab = lab[:,1:]
        batch["labels"] = lab
        return batch
collator = Collator(processor)

## 셀 7 — CER 지표 (전체 + 단어/문장 분리)

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")

# 에폭 평가에서 **단어/문장 CER을 따로** 낸다. 9차의 판정 지표가 dev 단어 CER이라
# 전체 CER만으로는 판단할 수 없다(단어가 세그먼트의 절반이지만 짧아서 전체
# CER에 묻힌다).
#
# 어느 스플릿인지는 **길이로** 가린다 — dev 326 vs test 417이라 겹치지 않는다.
# 재패키징으로 두 길이가 같아지면 이 분기가 조용히 틀리므로 아래에서 막는다.
TASK_TYPES = {}
for _name, _tt in (("dev", dev_task_types), ("test", test_task_types)):
    if not _tt:
        continue
    if len(_tt) in TASK_TYPES:
        raise SystemExit(f"dev와 test 길이가 같다({len(_tt)}) — 길이로 스플릿을 못 가른다.")
    TASK_TYPES[len(_tt)] = _tt

def compute_metrics(p):
    lab = p.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    pred = processor.tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
    ref  = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    out = {"cer": cer_metric.compute(predictions=pred, references=ref)}
    tt = TASK_TYPES.get(len(ref))
    if tt:
        for _short, _key in (("word", "wordlist"), ("sent", "narrative")):
            idx = [i for i, t in enumerate(tt) if t == _key]
            if idx:
                out[f"cer_{_short}"] = cer_metric.compute(
                    predictions=[pred[i] for i in idx],
                    references=[ref[i] for i in idx])
    return out

## 셀 5 — 어댑터별 홀드아웃 평가 + 판정

In [ ]:
import json, time, unicodedata, re, warnings
import transformers as _tf
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
_tf.logging.set_verbosity_error(); warnings.filterwarnings("ignore")

# 15종 중 14종은 이미 big6 학습 어휘에도 있다 — 7차재현·10차에게 "처음 보는
# 단어"가 아니다. 넷 다 공정한 단어는 사탕 하나뿐이다(대장 「12차 계획」).
BIG6_OVERLAP = {"가위", "거울", "꽃", "나무", "눈", "발", "배", "북", "사과",
                "신발", "연", "장갑", "풍선", "학교"}
FAIR_SPLITS = {"dev", "test"}   # 원래 이 스플릿이었던 행만 "어느 모델도 못 본 클립"이다

RESULT_PATH = "/content/drive/MyDrive/whisper-608-vochold-results.json"
PRED_PATH = "/content/drive/MyDrive/whisper-608-vochold-preds-{}.json"

# 사전 등록한 판정 문턱(대장 「12차 계획」, 결과 보기 전에 적었다)
FAIR_TOLERANCE = 0.10   # 원래 dev·test였던 행에서 12차가 11차보다 이 이상 나쁘면 안 된다


def _norm(t):
    t = unicodedata.normalize("NFC", t or "")
    t = re.sub(r"[^\w가-힣\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _lev(a, b):
    p = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        c = [i]
        for j, y in enumerate(b, 1):
            c.append(min(p[j] + 1, c[j - 1] + 1, p[j - 1] + (x != y)))
        p = c
    return p[-1]


try:
    import evaluate
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "evaluate", "jiwer"], check=True)
    import evaluate
_cer_metric = evaluate.load("cer")


def _cer(hyps, refs):
    if not refs:
        return float("nan")
    return _cer_metric.compute(predictions=hyps, references=refs)


def summarize(hyps, refs, words_, splits_):
    n = len(refs)
    idx_fair = [i for i in range(n) if splits_[i] in FAIR_SPLITS]
    idx_all = list(range(n))
    idx_candy_fair = [i for i in idx_fair if words_[i] == "사탕"]

    def grp(idx):
        if not idx:
            return None
        h, r = [hyps[i] for i in idx], [refs[i] for i in idx]
        return {"n": len(idx), "cer": _cer(h, r),
                "acc": sum(_norm(hyps[i]) == _norm(refs[i]) for i in idx) / len(idx)}

    out = {"fair": grp(idx_fair), "all": grp(idx_all), "사탕_공정": grp(idx_candy_fair)}
    by_word = {}
    for w in sorted(set(words_)):
        idx_w = [i for i in idx_fair if words_[i] == w]   # 단어별도 "공정"(dev·test) 기준
        by_word[w] = grp(idx_w)
    out["by_word"] = by_word
    return out


def verdict(new, ctl):
    """12차(new) 대 11차(ctl)를 사전 등록한 문턱으로 판정한다. 둘 다 summarize() 결과."""
    if new["fair"] is None or ctl["fair"] is None:
        return "n=0 — 판정 불가", float("nan")
    diff = new["fair"]["cer"] - ctl["fair"]["cer"]   # 양수 = 12차가 나쁘다
    if diff <= FAIR_TOLERANCE:
        v = f"어휘 몰라도 봐줄 만하다 (대가 {diff:+.4f} <= 문턱 {FAIR_TOLERANCE})"
    else:
        v = f"어휘 노출이 이득의 대부분이다 (대가 {diff:+.4f} > 문턱 {FAIR_TOLERANCE})"
    return v, diff


args = Seq2SeqTrainingArguments(
    output_dir="/content/vochold_tmp",
    per_device_eval_batch_size=8, dataloader_num_workers=2, fp16=True,
    predict_with_generate=True, remove_unused_columns=False,
    label_names=["labels"], report_to="none")

results = json.load(open(RESULT_PATH, encoding="utf-8")) if os.path.exists(RESULT_PATH) else {}
for name in found:
    if name in results:
        print(f"[{name}] 이미 잰 결과가 있다 — 건너뜀")
        continue
    model.set_adapter(name)
    assert model.active_adapter == name, f"어댑터가 안 바뀌었다: {model.active_adapter}"
    trainer = Seq2SeqTrainer(model=model, args=args, data_collator=collator,
                             compute_metrics=compute_metrics, processing_class=processor)
    t0 = time.time()
    pred = trainer.predict(ds["holdout"])
    lab = pred.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyps = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    refs = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    res = summarize(hyps, refs, words, orig_split)
    results[name] = res
    json.dump(results, open(RESULT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump([{"word": w, "orig_split": s, "ref": r, "hyp": h}
               for w, s, r, h in zip(words, orig_split, refs, hyps)],
              open(PRED_PATH.format(name), "w", encoding="utf-8"), ensure_ascii=False)
    fair = res["fair"]
    print(f"[{name}] {(time.time()-t0)/60:.1f}분 · 공정(dev+test, n={fair['n'] if fair else 0}) "
          f"CER {fair['cer'] if fair else float('nan'):.4f} · 전체(n={res['all']['n']}) "
          f"CER {res['all']['cer']:.4f}", flush=True)
    del trainer
    import gc
    gc.collect(); torch.cuda.empty_cache()

# ══ 요약 ═════════════════════════════════════════════════════
print()
print("  " + "어댑터".ljust(10) + "공정(dev+test)".rjust(16) + "전체(참고)".rjust(14) + "n(공정/전체)".rjust(16))
for name, r in results.items():
    f, a = r["fair"], r["all"]
    print("  " + name.ljust(10) +
          (f"{f['cer']:.4f}" if f else "-").rjust(16) +
          f"{a['cer']:.4f}".rjust(14) +
          f"({f['n'] if f else 0}/{a['n']})".rjust(16))

if "12차" in results and "11차" in results:
    v, diff = verdict(results["12차"], results["11차"])
    print(f"\n[판정] 12차 대 11차, 공정 부분집합(원래 dev·test였던 457개) → {v}")

if all(n in results for n in ("12차", "11차")):
    print("\n[사탕 — 넷 다 공정한 유일한 단어, 원래 dev·test였던 행만]")
    for name in ("12차", "11차", "7차재현", "10차"):
        if name in results and results[name]["사탕_공정"]:
            c = results[name]["사탕_공정"]
            print(f"  {name:10} CER {c['cer']:.4f} · 정확도 {c['acc']:.1%} (n={c['n']})")

print("\n[단어별 CER — 공정 부분집합, 통제군에게도 낯선 단어(사탕)만 별도 표시]")
words_seen = sorted(next(iter(results.values()))["by_word"])
print("  " + "단어".ljust(6) + "".join(n.rjust(12) for n in results) + "   big6에 있었나")
for w in words_seen:
    row = "  " + w.ljust(6)
    for name in results:
        bw = results[name]["by_word"].get(w)
        row += (f"{bw['cer']:.3f}" if bw else "-").rjust(12)
    row += "   " + ("있음(통제군도 앎)" if w in BIG6_OVERLAP else "없음(넷 다 낯섦)")
    print(row)
